# Phase 3d — Second external cohort: TNUI-2021 val (LG-PM external track, pre-declared)

**Why:** the TNUI-test external cohort (Phase 3c) has only **5 benign cases** (prev 0.964),
so T1c/T2c/T3c were power-limited. This notebook evaluates the untouched **TNUI-2021 val**
split as a pre-declared second external cohort and a **pre-declared pooled test+val
sensitivity**. This full analysis was specified before any TNUI-val label or metric was
inspected.

**Frozen decision rules (nothing refits on TNUI-val or pooled data):**
- Same frozen segmentation stack + same asserted K=20 support episode (as 3c);
- Linear heads **loaded from the Phase 3c cache** (identical heads across cohorts);
  tolerance-gated retrain (|Δ val AUROC| ≤ 0.01 vs Phase 3 originals) only if the
  cache is unavailable in this session;
- Thresholds + temperature from **TN5000 val only** (raw-raw) — computed in §6 from
  source probabilities, never touched by any TNUI number;
- ERM sanity gate: TN5000 test AUROC 0.9401 ± 0.002;
- **Replication gate:** TNUI-test AUROCs recomputed here must match the locked 3c table
  (±0.0015 with cached heads) — else pooled tables are withheld (POOLED_OK=False);
- Bonferroni α = 0.05/5 = 0.01 **per family** (val family T1d–T5d; pooled family P1–P5).

**Sanity expectations (printed, not fitted):** TNUI-val predicted-mask Dice near 0.708;
recomputed TNUI-test Dice 0.7046 ± 0.002 if the bank is recomputed.

## 0. Configuration, Phase 3 cache discovery, checkpoint discovery

In [ ]:
from pathlib import Path
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from IPython.display import display
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN = DEVICE.type == "cuda"

INPUT_ROOT  = Path("/kaggle/input/datasets")
AUTHOR_ROOT = INPUT_ROOT / "jonbon8397"
PATHS = {
    "tn3k": AUTHOR_ROOT / "tn3k-data" / "Thyroid Dataset" / "tn3k",
    "tnui": AUTHOR_ROOT / "tnui-2021" / "TNUI-2021-" / "thyroid_data",
    "tn5000": AUTHOR_ROOT / "tn5000" / "TN5000_forReview",
}
P3 = Path("/kaggle/working/phase3_classification")
P3C = Path("/kaggle/working/phase3c_external")
OUTPUT_ROOT = Path("/kaggle/working/phase3d_external")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# protocol constants identical to Phases 1-3
IMG_SIZE, MASK_THRESH, PRED_THRESH = 384, 127, 0.5
SEG_K, RING_PX = 20, 15
PROTO_SCALES, SCORE_HW = ["e2", "e3", "e4"], 48
FEAT_SCALES = ["e3", "e4"]
BATCH, N_BOOT = 16, 2000
ALPHA_BONF = 0.05 / 5
LIN_EPOCHS, LIN_PATIENCE, LIN_LR, LIN_WD = 300, 25, 3e-3, 1e-2

ck_enc = list(Path("/kaggle/input").rglob("best_resnet34_seed42.pt")) + \
         list(Path("/kaggle/working").rglob("best_resnet34_seed42.pt"))
ck_dec = list(Path("/kaggle/input").rglob("best_fusion_decoder.pt")) + \
         list(Path("/kaggle/working").rglob("best_fusion_decoder.pt"))
assert ck_enc and ck_dec, "attach the Phase 1/2 checkpoint Kaggle datasets"
CKPT_ENC, CKPT_DEC = ck_enc[0], ck_dec[0]

NEED = ["features_train.npz", "features_val.npz", "features_test.npz",
        "linear_heads_results.json", "erm_resnet34_tn5000.pt",
        "erm_resnet34_probs.json", "run_summary.json"]
miss = [f for f in NEED if not (P3 / f).exists()]
assert not miss, f"missing Phase 3 caches {miss} - run Phase 3 first in this working dir"
print("device:", DEVICE)
print("caches:", P3, "| 3c caches:", P3C, "(exists:", P3C.exists(), ") | output:", OUTPUT_ROOT)

## 1. TNUI-2021 test manifest with labels; TN5000 cross-dataset duplicate audit

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def build_tnui_split(split):
    imgs = {p.stem: p for p in image_files(PATHS["tnui"]/split/"images")}
    msks = {p.stem: p for p in image_files(PATHS["tnui"]/split/"masks")}
    common = sorted(set(imgs) & set(msks))
    return pd.DataFrame([{"id": k, "image": str(imgs[k]), "mask": str(msks[k])}
                         for k in common]), common

tnui_val, common_val = build_tnui_split("val")
tnui_test, common_test = build_tnui_split("test")
print(f"TNUI image-mask pairs: val={len(tnui_val)} | test={len(tnui_test)} (pooling only)")

# ---------------- label CSVs (tolerant parser, per split) ----------------
def load_tnui_labels_split(root, split):
    cands = sorted(Path(root).rglob("*labels*.csv"))
    rows = {}
    for cf in cands:
        if split not in cf.stem.lower():
            continue
        try:
            df = pd.read_csv(cf)
        except Exception:
            continue
        stem_col = None; lab_col = None
        for c in df.columns:
            vals = df[c].astype(str).str.strip().str.lower()
            if lab_col is None and set(vals.unique()) <= {"0", "1", "benign", "malignant"}:
                lab_col = c
        common = common_val if split == "val" else common_test
        for c in df.columns:
            if c == lab_col:
                continue
            vals = df[c].astype(str).map(lambda s: Path(s).stem)
            hit = sum(v in common for v in vals)
            if hit > max(10, int(0.5 * len(common))):
                stem_col = c
        if stem_col and lab_col:
            for _, r in df.iterrows():
                k = Path(str(r[stem_col])).stem
                v = str(r[lab_col]).strip().lower()
                if k in common and v in {"0", "1", "benign", "malignant"}:
                    rows[k] = 1 if v in {"1", "malignant"} else 0
    return rows

lab_val = load_tnui_labels_split(PATHS["tnui"], "val")
lab_test = load_tnui_labels_split(PATHS["tnui"], "test")

def attach_labels(df, lab_map, split):
    n_lab = df.id.map(lab_map).notna().sum()
    print(f"labels matched to {split} ids: {n_lab}")
    df = df[df.id.map(lab_map).notna()].copy()
    df["label"] = df.id.map(lab_map).astype(int)
    assert len(df) > 20, f"TNUI {split} label matching failed - inspect CSV columns"
    df = df.reset_index(drop=True)
    print(f"TNUI {split} usable: n={len(df)} | malignant={int(df.label.sum())} ({df.label.mean():.3f})")
    return df

tnui_val = attach_labels(tnui_val, lab_val, "val")
tnui_test = attach_labels(tnui_test, lab_test, "test")

# ---------------- audits: val vs TN5000 (train set), val vs test (disjointness) ----------------
AUD = OUTPUT_ROOT / "audit_hashes.json"
if AUD.exists():
    aud = json.loads(AUD.read_text())
else:
    tnui_val["sha256"] = tnui_val["image"].apply(sha256_file)
    test_hashes = {sha256_file(p) for p in tnui_test["image"]}
    t5_hashes = {sha256_file(p) for p in image_files(PATHS["tn5000"]/"JPEGImages")}
    aud = {"tnui_val_hashes": tnui_val.set_index("id").sha256.to_dict(),
           "overlap_tn5000": sorted(set(tnui_val.sha256) & t5_hashes),
           "overlap_tnui_test": sorted(set(tnui_val.sha256) & test_hashes),
           "within_val_dups": int(tnui_val.sha256.duplicated().sum())}
    AUD.write_text(json.dumps(aud, indent=1))
ov5, ovt = aud["overlap_tn5000"], aud["overlap_tnui_test"]
if "sha256" not in tnui_val.columns:
    tnui_val["sha256"] = tnui_val.id.map(aud["tnui_val_hashes"])
bad = set(ov5) | set(ovt)
if bad:
    tnui_val = tnui_val[~tnui_val.sha256.isin(bad)].reset_index(drop=True)
if aud["within_val_dups"]:
    tnui_val = tnui_val.drop_duplicates("sha256", keep="first").reset_index(drop=True)
print(f"TNUI-val vs TN5000 exact-dups: {len(ov5)} (excluded) | "
      f"val vs test exact-dups: {len(ovt)} (excluded) | within-val dups: {aud['within_val_dups']} (dropped)")

## 2. Loaders (identical to Phase 3) + fixed support episode cross-check

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def read_gray(p):
    return np.asarray(Image.open(p).convert("L"))

def letterbox_image(img, size=IMG_SIZE):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    ri = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    t, l = (size - nh) // 2, (size - nw) // 2
    ci = np.zeros((size, size), np.float32); ci[t:t + nh, l:l + nw] = ri / 255.0
    return ci, {"s": s, "t": t, "l": l, "nh": nh, "nw": nw}

def letterbox_mask(msk_raw, size=IMG_SIZE):
    h, w = msk_raw.shape[:2]
    s = size / max(h, w)
    nh, nw = max(1, int(round(h * s))), max(1, int(round(w * s)))
    rm = cv2.resize(msk_raw, (nw, nh), interpolation=cv2.INTER_NEAREST)
    t, l = (size - nh) // 2, (size - nw) // 2
    cm = np.zeros((size, size), np.uint8); cm[t:t + nh, l:l + nw] = (rm > 0).astype(np.uint8)
    return cm

def make_ring(mask, r=RING_PX):
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    dil = cv2.dilate(mask.astype(np.uint8), k)
    ring = np.clip(dil.astype(np.int16) - mask.astype(np.int16), 0, 1).astype(np.uint8)
    if ring.sum() == 0:
        ring = (1 - mask.astype(np.uint8))
    return ring

def to_tensor(img):
    x = torch.from_numpy(np.stack([img, img, img])).float()
    return (x - MEAN) / STD

# -------- TN3K train pool and THE SAME support episode as Phase 3 --------
def pair_images_masks(image_dir, mask_dir, dataset, split):
    im = {p.stem: p for p in image_files(image_dir)}
    ms = {p.stem: p for p in image_files(mask_dir)}
    cm = sorted(set(im) & set(ms))
    return pd.DataFrame([{"dataset": dataset, "split": split, "id": k,
                          "image": str(im[k]), "mask": str(ms[k])} for k in cm])

tn3k = pair_images_masks(PATHS["tn3k"]/"trainval-image", PATHS["tn3k"]/"trainval-mask", "TN3K", "trainval")
tn3k["sha256"] = tn3k["image"].apply(sha256_file)
tn3k = tn3k.sort_values("id", kind="stable").drop_duplicates("sha256", keep="first")
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(tn3k))
val_ids = set(tn3k.iloc[perm[: int(round(0.15 * len(tn3k)))]].id)
tn3k["split"] = np.where(tn3k.id.isin(val_ids), "val", "train")
tn3k_train = tn3k[tn3k.split == "train"].reset_index(drop=True)

def tn3k_support_rows(kmax=SEG_K, seed=SEED + 10):
    r = np.random.RandomState(seed)
    return tn3k_train.iloc[r.permutation(len(tn3k_train))[:kmax]].reset_index(drop=True)

SUP_ROWS = tn3k_support_rows()
_arch = json.loads((P3/"run_summary.json").read_text())["config"]["support_episode_ids"]
assert list(SUP_ROWS.id) == _arch, "support episode mismatch vs Phase 3 archive - seed drift!"
print(f"support episode reproduced exactly ({SEG_K} ids match Phase 3 archive); pool n={len(tn3k_train)}")

def load_tn3k_item(row):
    img = read_gray(row["image"]).astype(np.float32)
    ci, _ = letterbox_image(img)
    cm = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    return ci, cm

## 3. Frozen LG-PM components (exact copies from Phase 3)

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = conv_block(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class Phase1UNet(nn.Module):
    def __init__(self):
        super().__init__()
        import torchvision.models as tvm
        bb = tvm.resnet34(weights=None)
        self.stem = nn.Sequential(bb.conv1, bb.bn1, bb.relu)
        self.pool = bb.maxpool
        self.enc1, self.enc2, self.enc3, self.enc4 = bb.layer1, bb.layer2, bb.layer3, bb.layer4
        self.up4, self.up3 = Up(512, 256, 256), Up(256, 128, 128)
        self.up2, self.up1 = Up(128, 64, 64), Up(64, 64, 64)
        self.head = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            conv_block(64, 32), nn.Conv2d(32, 1, 1))
    def features(self, x):
        e0 = self.stem(x)
        e1 = self.enc1(self.pool(e0)); e2 = self.enc2(e1)
        e3 = self.enc3(e2); e4 = self.enc4(e3)
        return {"e1": e1, "e2": e2, "e3": e3, "e4": e4}

def load_ckpt(path):
    # torch >= 2.6 defaults torch.load(..., weights_only=True); our own checkpoints
    # may embed numpy scalars in metadata (ERM ckpt val-AUROC), which the
    # restricted unpickler rejects. Try the restricted loader first, then fall
    # back explicitly -- files are our own trusted artifacts.
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu", weights_only=False)

backbone = Phase1UNet().to(DEVICE)
_ck = load_ckpt(CKPT_ENC)
backbone.load_state_dict(_ck["model"], strict=True)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def encode(x):
    with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
        f = backbone.features(x)
    return {k: v.float() for k, v in f.items()}
print("encoder loaded (val Dice:", round(float(_ck.get("val_dice")), 4), ")")

In [ ]:
def masked_proto(feats, masks):
    h, w = feats.shape[-2:]
    m = F.interpolate(masks, size=(h, w), mode="nearest")
    fg_den = m.sum(dim=(0, 2, 3)) + 1e-6
    bg_den = (1 - m).sum(dim=(0, 2, 3)) + 1e-6
    pf = (feats * m).sum(dim=(0, 2, 3)) / fg_den
    pb = (feats * (1 - m)).sum(dim=(0, 2, 3)) / bg_den
    return pf, pb

def neg_sq_dist(feats, proto):
    return -(feats - proto.view(1, -1, 1, 1)).pow(2).sum(1, keepdim=True)

def score_maps(que_feats, pf_pb_per_scale):
    outs = []
    for name in PROTO_SCALES:
        qf = que_feats[name]
        pf, pb = pf_pb_per_scale[name]
        outs.append(F.interpolate(neg_sq_dist(qf, pf), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
        outs.append(F.interpolate(neg_sq_dist(qf, pb), size=(SCORE_HW, SCORE_HW),
                                  mode="bilinear", align_corners=False))
    return torch.cat(outs, dim=1)

class FusionDecoder(nn.Module):
    def __init__(self, cin=2 * len(PROTO_SCALES)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1))
    def forward(self, s):
        return self.net(s)

def predict_logits_384(que_feats, pf_pb_per_scale, decoder):
    return F.interpolate(decoder(score_maps(que_feats, pf_pb_per_scale)),
                         size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

decoder = FusionDecoder().to(DEVICE)
_dk = load_ckpt(CKPT_DEC)
decoder.load_state_dict(_dk["decoder"], strict=True)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

_sup = [load_tn3k_item(r) for _, r in SUP_ROWS.iterrows()]
sup_x = torch.stack([to_tensor(i) for i, _ in _sup]).to(DEVICE)
sup_m = torch.stack([torch.from_numpy(m[None]) for _, m in _sup]).float().to(DEVICE)
with torch.no_grad():
    SF = encode(sup_x)
PF_PB = {name: masked_proto(SF[name], sup_m) for name in PROTO_SCALES}
print("decoder loaded | fixed support prototypes ready (3 scales)")

## 4. TNUI feature bank — zero-shot segmentation + descriptors + mask QC (Dice)
Oracle rows use **true TNUI masks** (stronger oracle than TN5000’s expert box).

In [ ]:
def dice_np(a, b, eps=1e-7):
    a, b = (a > 0).astype(np.float32), (b > 0).astype(np.float32)
    inter = (a * b).sum()
    return float((2 * inter + eps) / (a.sum() + b.sum() + eps))

def masked_gap(feat, msk01):
    h, w = feat.shape[-2:]
    m = F.interpolate(msk01, size=(h, w), mode="nearest")
    den = m.sum(dim=(2, 3)).clamp(min=1.0)
    return (feat * m).sum(dim=(2, 3)) / den

REPS = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtmask", "R2g_gtmask_ring"]
def run_cohort_bank(df, tag):
    fpath = OUTPUT_ROOT / f"features_tnui_{tag}.npz"
    if fpath.exists():
        z = np.load(fpath, allow_pickle=True)
        print(f"[cache] TNUI {tag} feature bank loaded")
        return {k: z[k] for k in z.files}
    store = {r: [] for r in REPS}
    mask96, conf, area, empty, ids, labels, dices = [], [], [], [], [], [], []
    t0 = time.time()
    for i0 in range(0, len(df), BATCH):
        chunk = df.iloc[i0:i0 + BATCH]
        imgs, gts = [], []
        for _, r in chunk.iterrows():
            ci, _ = letterbox_image(read_gray(r["image"]).astype(np.float32))
            imgs.append(ci)
            gts.append(letterbox_mask((read_gray(r["mask"]) >= MASK_THRESH).astype(np.uint8)))
        x = torch.stack([to_tensor(i) for i in imgs]).to(DEVICE)
        with torch.no_grad():
            feats = encode(x)
            prob = torch.sigmoid(predict_logits_384(feats, PF_PB, decoder))
        pm = (prob >= PRED_THRESH).float()
        pm_raw = pm.clone()
        for bi in range(len(chunk)):
            if pm[bi].sum() < 4:
                pm[bi] = 1.0; empty.append(1)
            else:
                empty.append(0)
        gtm = torch.from_numpy(np.stack(gts)[:, None]).float().to(DEVICE)
        ring_p = torch.stack([torch.from_numpy(make_ring(m[0].cpu().numpy().astype(np.uint8))[None])
                              for m in pm]).float().to(DEVICE)
        ring_g = torch.stack([torch.from_numpy(make_ring(gts[bi])[None])
                              for bi in range(len(chunk))]).float().to(DEVICE)
        combo = {"R0_image": "image",
                 "R1p_lesion": pm, "R2p_lesion_ring": (pm, ring_p),
                 "R1g_gtmask": gtm, "R2g_gtmask_ring": (gtm, ring_g)}
        desc = {}
        for rep, key in combo.items():
            if key == "image":
                desc[rep] = torch.cat([feats[nm].mean(dim=(2, 3)) for nm in FEAT_SCALES], 1).float().cpu()
            elif isinstance(key, tuple):
                a = torch.cat([masked_gap(feats[nm], key[0]) for nm in FEAT_SCALES], 1)
                b = torch.cat([masked_gap(feats[nm], key[1]) for nm in FEAT_SCALES], 1)
                desc[rep] = torch.cat([a, b], 1).float().cpu()
            else:
                desc[rep] = torch.cat([masked_gap(feats[nm], key) for nm in FEAT_SCALES], 1).float().cpu()
        for r_ in REPS:
            store[r_].append(desc[r_].numpy())
        for bi in range(len(chunk)):
            mask96.append(cv2.resize(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), (96, 96),
                                     interpolation=cv2.INTER_NEAREST))
            conf.append(float(prob[bi].mean())); area.append(float(pm_raw[bi].mean()))
            dices.append(dice_np(pm_raw[bi, 0].cpu().numpy().astype(np.uint8), gts[bi]))
        ids.extend(chunk.id.tolist()); labels.extend(chunk.label.astype(int).tolist())
        done = min(i0 + BATCH, len(df))
        if done % (BATCH * 25) < BATCH or done == len(df):
            print(f"  tnui-{tag}: {done}/{len(df)} images ({(time.time()-t0)/60:.1f} min)")
    out = {"ids": np.array(ids), "labels": np.array(labels, np.int64),
           "mask96": np.stack(mask96), "conf": np.array(conf, np.float32),
           "area": np.array(area, np.float32), "empty": np.array(empty, np.int64),
           "dice": np.array(dices, np.float32)}
    for r_ in REPS:
        out[r_] = np.concatenate(store[r_]).astype(np.float32)
    np.savez_compressed(fpath, **out)
    print(f"[done] tnui-{tag} n={len(ids)} | mean seg Dice {np.mean(dices):.4f} "
          f"(Phase 2 zero-shot K=20 ref 0.708) | empty-mask {sum(empty)} ({100*sum(empty)/len(ids):.1f}%) "
          f"| mean area {np.mean(area):.3f} | {(time.time()-t0)/60:.1f} min")
    return out

TNUI_VAL_BANK = run_cohort_bank(tnui_val, "val")

# TNUI-test bank for the PRE-DECLARED pooled sensitivity only: reuse the 3c cache when
# available (bit-identical to the locked run), else recompute with identical code.
P3C_BANK = P3C / "features_tnui_test.npz"
if P3C_BANK.exists():
    _z = np.load(P3C_BANK, allow_pickle=True)
    TNUI_TEST_BANK = {k: _z[k] for k in _z.files}
    TEST_BANK_SRC = "3c cache (locked-run bytes)"
else:
    TNUI_TEST_BANK = run_cohort_bank(tnui_test, "test")
    TEST_BANK_SRC = "recomputed in 3d (identical code)"
    assert abs(np.mean(TNUI_TEST_BANK["dice"]) - 0.7046) <= 0.002, \
        "recomputed test bank deviates from locked 3c Dice - investigate before pooling"
print("TNUI-test bank provenance:", TEST_BANK_SRC, "| mean dice:",
      round(float(np.mean(TNUI_TEST_BANK["dice"])), 4))

## 5. Source-fitted classifiers — retrained linear heads (tolerance-gated) + ERM (sanity-gated)
Linear heads are re-derived from cached TN5000 features under the Phase 3 protocol.
Exact weight reproduction is not promised (RNG after ERM differs) — instead we gate on
val AUROC within 0.01 of the Phase 3 originals and report both. ERM must re-hit
0.9401 ± 0.002 on TN5000 test, else the external row is not trusted.

In [ ]:
P3B = {s: {k: v for k, v in np.load(P3 / f"features_{s}.npz", allow_pickle=True).items()}
       for s in ["train", "val", "test"]}
LIN_ORIG = json.loads((P3/"linear_heads_results.json").read_text())

def feat(rep, split):
    return torch.from_numpy(P3B[split][rep].astype(np.float32))

def labels_np(split):
    return P3B[split]["labels"].astype(np.float32)

def train_linear(rep):
    xtr, xva, xte = feat(rep, "train"), feat(rep, "val"), feat(rep, "test")
    ytr_np = labels_np("train")
    mu, sd = xtr.mean(0, keepdim=True), xtr.std(0, keepdim=True).clamp(min=1e-6)
    xtr, xva, xte = (xtr-mu)/sd, (xva-mu)/sd, (xte-mu)/sd
    xtr, xva, xte = xtr.to(DEVICE), xva.to(DEVICE), xte.to(DEVICE)
    ytr = torch.from_numpy(ytr_np).to(DEVICE)
    lin = nn.Linear(xtr.shape[1], 1).to(DEVICE)
    pos_w = torch.tensor([(len(ytr_np)-ytr_np.sum())/max(ytr_np.sum(), 1)], device=DEVICE)
    opt = torch.optim.AdamW(lin.parameters(), lr=LIN_LR, weight_decay=LIN_WD)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    best, wait, bw = -1, 0, None
    n = len(xtr)
    for ep in range(1, LIN_EPOCHS + 1):
        perm = torch.randperm(n, device=DEVICE)
        lin.train()
        for i0 in range(0, n, 512):
            idx = perm[i0:i0+512]
            loss = bce(lin(xtr[idx]).squeeze(1), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        lin.eval()
        with torch.no_grad():
            pv = torch.sigmoid(lin(xva).squeeze(1)).cpu().numpy()
        va = roc_auc_score(labels_np("val"), pv)
        if va > best + 1e-4:
            best, wait = va, 0
            bw = {k: v.detach().cpu().clone() for k, v in lin.state_dict().items()}
        else:
            wait += 1
            if wait >= LIN_PATIENCE:
                break
    lin.load_state_dict(bw); lin.eval()
    with torch.no_grad():
        out = {s: torch.sigmoid(lin(x).squeeze(1)).cpu().numpy()
               for s, x in [("val", xva), ("test", xte)]}
    head = {"w": bw["weight"].numpy(), "b": bw["bias"].numpy(),
            "mu": mu.numpy(), "sd": sd.numpy()}
    return {"val_auroc": float(best), "probs": out, "head": head}

REPS_SRC = ["R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtbox", "R2g_gtbox_ring"]
MAP_EXT = {"R0_image": "R0_image", "R1p_lesion": "R1p_lesion",
           "R2p_lesion_ring": "R2p_lesion_ring", "R1g_gtbox": "R1g_gtmask",
           "R2g_gtbox_ring": "R2g_gtmask_ring"}
LINW_3C = P3C / "linear_heads_retrained.json"
LINW_PATH = OUTPUT_ROOT / "linear_heads_retrained.json"
HEADS_SRC = None
if LINW_3C.exists():
    LINR = json.loads(LINW_3C.read_text())
    HEADS_SRC = "phase3c cache (identical heads across cohorts - the pre-declaration)"
    print("[cache] 3c heads loaded:", HEADS_SRC)
elif LINW_PATH.exists():
    LINR = json.loads(LINW_PATH.read_text())
    HEADS_SRC = "3d-local retrained cache"
    print("[cache] 3d-local retrained heads loaded")
else:
    HEADS_SRC = "retrained in 3d (tolerance-gated)"
    LINR = {}
    for rep in REPS_SRC:
        LINR[rep] = train_linear(rep)
        o = LIN_ORIG[rep]["val_auroc"]
        d = LINR[rep]["val_auroc"] - o
        flag = "OK" if abs(d) <= 0.01 else "OUT-OF-TOLERANCE"
        print(f"{rep:16s} val AUROC {LINR[rep]['val_auroc']:.4f} (orig {o:.4f}, delta {d:+.4f}) {flag}")
        LINR[rep]["probs"] = {s: v.tolist() for s, v in LINR[rep]["probs"].items()}
        LINR[rep]["head"] = {k: v.tolist() for k, v in LINR[rep]["head"].items()}
    LINW_PATH.write_text(json.dumps(LINR, indent=1))
print("heads provenance:", HEADS_SRC)

def lin_apply(rep, X):
    h = LINR[rep]["head"]
    w = np.array(h["w"]); b = np.array(h["b"]); mu = np.array(h["mu"]); sd = np.array(h["sd"])
    z = ((X.astype(np.float64) - mu) / sd) @ w.T + b
    return (1 / (1 + np.exp(-z.ravel()))).astype(float)

# ERM sanity gate
import torchvision.models as tvm
erm = tvm.resnet34(weights=None); erm.fc = nn.Linear(erm.fc.in_features, 1)
erm.load_state_dict(load_ckpt(P3/"erm_resnet34_tn5000.pt")["model"])
erm = erm.to(DEVICE).eval()

class ImgDS(Dataset):
    def __init__(self, df): self.df = df
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        ci, _ = letterbox_image(read_gray(self.df.iloc[i]["image"]).astype(np.float32))
        return to_tensor(ci), np.float32(self.df.iloc[i]["label"])

# TN5000 test sanity: rebuild its image rows from cached order? ids are in features npz
# ERM json probs are already aligned with TN5000 splits (saved in Phase 3) -> use them as truth
@torch.no_grad()
def erm_probs(df):
    ld = DataLoader(ImgDS(df), batch_size=32, shuffle=False, num_workers=2, pin_memory=PIN)
    ps = []
    for x, _ in ld:
        x = x.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            ps.append(torch.sigmoid(erm(x)).squeeze(1).float().cpu())
    return torch.cat(ps).numpy()

p_erm_val = erm_probs(tnui_val)
p_erm_test = erm_probs(tnui_test)
print(f"ERM probs: TNUI-val n={len(p_erm_val)} | TNUI-test n={len(p_erm_test)}")

## 6. External metrics on TNUI test — source-fitted thresholds and temperature

In [ ]:
def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def fit_temperature(y, p):
    z, yv = logit(np.asarray(p)), np.asarray(y, np.float64)
    grid = np.exp(np.linspace(np.log(0.05), np.log(20.0), 200))
    def nll(T):
        q = 1 / (1 + np.exp(-z / T))
        return -np.mean(yv * np.log(q + 1e-12) + (1 - yv) * np.log(1 - q + 1e-12))
    return float(grid[int(np.argmin([nll(T) for T in grid]))])

def ece(y, p, bins=15):
    y = np.asarray(y, np.float64).ravel(); p = np.asarray(p, np.float64).ravel()
    edges = np.linspace(0, 1, bins + 1); out = 0.0
    for i in range(bins):
        m = (p >= edges[i]) & (p < edges[i + 1] if i < bins - 1 else p <= edges[i + 1])
        if m.sum():
            out += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(out)

def spec90_threshold(y_val, p_val):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    spec = 1 - fpr
    m = spec >= 0.90
    if m.any():
        i = np.where(m)[0][np.argmax(tpr[m])]
    else:
        i = int(np.argmax(spec))
    return float(thr[i]), float(tpr[i]), float(spec[i])

def brier(y, p):
    return float(np.mean((np.asarray(p).ravel() - np.asarray(y).ravel()) ** 2))

def metric_set(y, p, thr):
    y = np.asarray(y).ravel(); p = np.asarray(p).ravel()
    return {"auroc": roc_auc_score(y, p), "auprc": average_precision_score(y, p),
            "sens@spec90": float((p[y == 1] >= thr).mean()), "brier": brier(y, p),
            "ece": ece(y, p)}

def boot_ci(y, p, thr, n=N_BOOT, seed=SEED):
    y, p = np.asarray(y).ravel(), np.asarray(p).ravel()
    rng = np.random.RandomState(seed); N = len(y); acc = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2:
            continue
        acc.append([roc_auc_score(y[idx], p[idx]), average_precision_score(y[idx], p[idx]),
                    (p[idx][y[idx] == 1] >= thr).mean()])
    lo, hi = np.percentile(acc, 2.5, 0), np.percentile(acc, 97.5, 0)
    return lo, hi

MODELS = ["ERM", "R0_image", "R1p_lesion", "R2p_lesion_ring", "R1g_gtmask", "R2g_gtmask_ring"]

def cohort_probs(bank, p_erm):
    P = {"ERM": np.asarray(p_erm).ravel()}
    for rep_src, rep_ext in MAP_EXT.items():
        P[rep_ext] = lin_apply(rep_src, bank[rep_ext])
    return P

y_val = TNUI_VAL_BANK["labels"].astype(int)
y_test = TNUI_TEST_BANK["labels"].astype(int)
P_VAL = cohort_probs(TNUI_VAL_BANK, p_erm_val)
P_TEST = cohort_probs(TNUI_TEST_BANK, p_erm_test)

# replication gate: recomputed TNUI-test AUROCs vs LOCKED 3c table
LOCK3C = {"ERM": 0.5172, "R0_image": 0.5791, "R1p_lesion": 0.7119,
          "R2p_lesion_ring": 0.6776, "R1g_gtmask": 0.8343, "R2g_gtmask_ring": 0.8507}
_hetol = 0.0015 if HEADS_SRC.startswith("phase3c cache") else 0.01
_drift = {m: abs(roc_auc_score(y_test, P_TEST[m]) - LOCK3C[m]) for m in MODELS}
POOLED_OK = all(v <= _hetol for v in _drift.values())
print("replication gate vs locked 3c (tol", _hetol, "):",
      {m: round(v, 4) for m, v in _drift.items()}, "-> POOLED_OK" if POOLED_OK else "-> POOLED WITHHELD")

# source val probs for thresholds/temperature
ERMJ = json.loads((P3/"erm_resnet34_probs.json").read_text())
P_SRCV = {"ERM": np.array(ERMJ["probs"]["val"]).ravel()}
for rep_src in REPS_SRC:
    P_SRCV[MAP_EXT[rep_src]] = np.array(LINR[rep_src]["probs"]["val"]).ravel()
y_srcv = labels_np("val").astype(int)

# ERM sanity gate on TN5000 test
erm_ref = roc_auc_score(P3B["test"]["labels"].astype(int), np.array(ERMJ["probs"]["test"]).ravel())
assert abs(erm_ref - 0.9401) <= 0.002, f"ERM sanity failed: {erm_ref:.4f}"
print(f"ERM TN5000 test AUROC sanity: {erm_ref:.4f} (ref 0.9401) OK")

rows, THR_EXT, TEMP_EXT = [], {}, {}
for m in MODELS:
    pv_src = P_SRCV[m]; pe = P_VAL[m]
    T = fit_temperature(y_srcv, pv_src); TEMP_EXT[m] = T
    pec = 1 / (1 + np.exp(-logit(pe) / T))
    thr, s_v, sp_v = spec90_threshold(y_srcv, pv_src); THR_EXT[m] = thr
    ms = metric_set(y_val, pe, thr)
    ms["brier"] = brier(y_val, pec); ms["ece"] = ece(y_val, pec)
    lo, hi = boot_ci(y_val, pe, thr)
    rows.append({"model": m, "auroc": round(ms["auroc"], 4),
                 "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})", "auprc": round(ms["auprc"], 4),
                 "sens@spec90(TN5000val)": round(ms["sens@spec90"], 4),
                 "brier": round(ms["brier"], 4), "ece_post": round(ms["ece"], 4),
                 "ece_pre": round(ece(y_val, pe), 4), "T_srcval": round(T, 3)})
EXT = pd.DataFrame(rows)
EXT.to_csv(OUTPUT_ROOT / "phase3d_tnui_val_main_table.csv", index=False)
print(f"TNUI-2021 VAL (n={len(y_val)}, malignant={int(y_val.sum())}, benign={int((1-y_val).sum())}) "
      f"- all thresholds/temp from TN5000 val:")
display(EXT)

## 7. Domain-shift comparison and the five planned paired tests

In [ ]:
def delta_pair_boot(y, pa, pb, n=5000, seed=SEED):
    y = np.asarray(y); rng = np.random.RandomState(seed); N = len(y); d = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        if len(np.unique(y[idx])) < 2: continue
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(2*min((d<=0).mean(), (d>=0).mean()), 1.0), len(d)

def paired_boot_sens(y, pa, ta, pb, tb, n=5000, seed=SEED):
    y = np.asarray(y); pos = y == 1
    rng = np.random.RandomState(seed)
    sa = (pa[pos] >= ta).astype(float); sb = (pb[pos] >= tb).astype(float)
    M = len(sa); d = []
    for _ in range(n):
        idx = rng.randint(0, M, M)
        d.append(sa[idx].mean() - sb[idx].mean())
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), min(2*min((d<=0).mean(), (d>=0).mean()), 1.0), len(d)

tests = [
    ("T1d AUROC R1p - R0 (grounding, val cohort)", "auroc", "R1p_lesion", "R0_image"),
    ("T2d AUROC ERM - R1p (fine-tune advantage, val cohort)", "auroc", "ERM", "R1p_lesion"),
    ("T3d AUROC R1g(mask) - R1p (error propagation, val cohort)", "auroc", "R1g_gtmask", "R1p_lesion"),
    ("T4d AUROC ERM - R1g(mask) (fine-tune vs true oracle, val cohort)", "auroc", "ERM", "R1g_gtmask"),
    ("T5d sens@spec90 ERM - R1p (operating point, val cohort)", "sens", "ERM", "R1p_lesion"),
]
rows = []
for name, kind, a, b in tests:
    if kind == "auroc":
        m_, lo, hi, p, n = delta_pair_boot(y_val, P_VAL[a], P_VAL[b])
    else:
        m_, lo, hi, p, n = paired_boot_sens(y_val, P_VAL[a], THR_EXT[a], P_VAL[b], THR_EXT[b])
    rows.append({"test": name, "delta(A-B)": round(m_, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                 "p": round(p, 5), "sig(0.01)": "YES" if p < ALPHA_BONF else "no", "n_boot": n})
SIG = pd.DataFrame(rows)
SIG.to_csv(OUTPUT_ROOT / "phase3d_paired_tests_val.csv", index=False)
print(f"Bonferroni alpha = {ALPHA_BONF}")
display(SIG)

# 3-cohort shift table (rank family): TN5000 test / TNUI test / TNUI val
src_rows = {"ERM": np.array(ERMJ["probs"]["test"]).ravel()}
for rep_src, rep_ext in MAP_EXT.items():
    src_rows[rep_ext] = np.array(LINR[rep_src]["probs"]["test"]).ravel()
y_srct = P3B["test"]["labels"].astype(int)
shift = []
for m in MODELS:
    a_src = roc_auc_score(y_srct, src_rows[m])
    a_ite = roc_auc_score(y_test, P_TEST[m]); a_iva = roc_auc_score(y_val, P_VAL[m])
    shift.append({"model": m, "AUROC TN5000-test": round(a_src, 4),
                  "AUROC TNUI-test": round(a_ite, 4), "AUROC TNUI-val": round(a_iva, 4),
                  "drop test": round(a_src - a_ite, 4), "drop val": round(a_src - a_iva, 4)})
SH = pd.DataFrame(shift)
SH.to_csv(OUTPUT_ROOT / "phase3d_domain_shift_3cohort.csv", index=False)
print("domain shift (AUROC, rank metric - same models, three cohorts):")
display(SH)

## 7b. PRE-DECLARED pooled sensitivity — TNUI test+val, frozen thresholds (no refit)
Specified before any TNUI-val number was inspected. Pooled tables are shown only if the
replication gate passed (POOLED_OK). Thresholds/temperature remain the TN5000-val-derived
values from §6 — this cell must contain **no** `spec90_threshold`/`fit_temperature` calls.

In [ ]:
if not POOLED_OK:
    print("POOLED WITHHELD: replication gate failed - see Section 6 drift table.")
else:
    y_pool = np.concatenate([y_test, y_val])
    P_POOL = {m: np.concatenate([P_TEST[m], P_VAL[m]]) for m in MODELS}
    print(f"Pooled TNUI external (test+val): n={len(y_pool)} | malignant={int(y_pool.sum())} "
          f"({y_pool.mean():.3f}) | benign={int((1-y_pool).sum())}")
    rows = []
    for m in MODELS:
        pe = P_POOL[m]
        pec = 1 / (1 + np.exp(-logit(pe) / TEMP_EXT[m]))
        thr = THR_EXT[m]
        ms = metric_set(y_pool, pe, thr)
        ms["brier"] = brier(y_pool, pec); ms["ece"] = ece(y_pool, pec)
        lo, hi = boot_ci(y_pool, pe, thr)
        rows.append({"model": m, "auroc": round(ms["auroc"], 4),
                     "auroc_CI": f"({lo[0]:.4f}-{hi[0]:.4f})", "auprc": round(ms["auprc"], 4),
                     "sens@spec90(TN5000val)": round(ms["sens@spec90"], 4),
                     "brier": round(ms["brier"], 4), "ece_post": round(ms["ece"], 4)})
    POOL = pd.DataFrame(rows)
    POOL.to_csv(OUTPUT_ROOT / "phase3d_pooled_main_table.csv", index=False)
    print("POOLED SENSITIVITY (pre-declared) - frozen thresholds/temp from TN5000 val:")
    display(POOL)
    ptests = [
        ("P1 AUROC R1p - R0 (grounding, pooled)", "auroc", "R1p_lesion", "R0_image"),
        ("P2 AUROC ERM - R1p (fine-tune advantage, pooled)", "auroc", "ERM", "R1p_lesion"),
        ("P3 AUROC R1g(mask) - R1p (error propagation, pooled)", "auroc", "R1g_gtmask", "R1p_lesion"),
        ("P4 AUROC ERM - R1g(mask) (fine-tune vs oracle, pooled)", "auroc", "ERM", "R1g_gtmask"),
        ("P5 sens@spec90 ERM - R1p (operating point, pooled)", "sens", "ERM", "R1p_lesion"),
    ]
    rows = []
    for name, kind, a, b in ptests:
        if kind == "auroc":
            m_, lo, hi, p, n = delta_pair_boot(y_pool, P_POOL[a], P_POOL[b])
        else:
            m_, lo, hi, p, n = paired_boot_sens(y_pool, P_POOL[a], THR_EXT[a], P_POOL[b], THR_EXT[b])
        rows.append({"test": name, "delta(A-B)": round(m_, 4), "CI95": f"({lo:.4f}..{hi:.4f})",
                     "p": round(p, 5), "sig(0.01)": "YES" if p < ALPHA_BONF else "no", "n_boot": n})
    PSIG = pd.DataFrame(rows)
    PSIG.to_csv(OUTPUT_ROOT / "phase3d_pooled_paired_tests.csv", index=False)
    print(f"pooled paired tests (Bonferroni alpha = {ALPHA_BONF}, own family of 5):")
    display(PSIG)

## 8. Qualitative panel — TNUI test: image, true mask, predicted mask+ring, calibrated p(mal)

In [ ]:
rng_q = np.random.RandomState(SEED + 22)
pos_idx = tnui_val.index[tnui_val.label == 1].to_numpy()
neg_idx = tnui_val.index[tnui_val.label == 0].to_numpy()
n_pos, n_neg = min(3, len(pos_idx)), min(3, len(neg_idx))
pick = list(rng_q.choice(pos_idx, n_pos, replace=False)) + list(rng_q.choice(neg_idx, n_neg, replace=False))
id2row = {b: i for i, b in enumerate(TNUI_VAL_BANK["ids"])}
p_q = 1/(1+np.exp(-logit(P_VAL["R1p_lesion"])/TEMP_EXT["R1p_lesion"]))
fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.1 * len(pick)))
for r_, i_ in enumerate(pick):
    row = tnui_val.loc[i_]
    ci, _ = letterbox_image(read_gray(row["image"]).astype(np.float32))
    gt = letterbox_mask((read_gray(row["mask"]) >= MASK_THRESH).astype(np.uint8))
    m = cv2.resize(TNUI_VAL_BANK["mask96"][id2row[row.id]], (IMG_SIZE, IMG_SIZE),
                   interpolation=cv2.INTER_NEAREST)
    ring = make_ring(m.astype(np.uint8))
    def overlay(base, red, blue):
        o = np.dstack([base, base, base]).copy()
        o[..., 0] = np.clip(o[..., 0] + 0.5 * red, 0, 1)
        o[..., 2] = np.clip(o[..., 2] + 0.35 * blue, 0, 1)
        return o
    axes[r_, 0].imshow(ci, cmap="gray"); axes[r_, 0].set_title(f"{row.id} label={int(row.label)}")
    axes[r_, 1].imshow(overlay(ci, gt, gt * 0)); axes[r_, 1].set_title("true mask (oracle)")
    axes[r_, 2].imshow(overlay(ci, m, ring))
    axes[r_, 2].set_title(f"pred mask+ring, p={p_q[id2row[row.id]]:.2f}, "
                          f"dice={TNUI_VAL_BANK['dice'][id2row[row.id]]:.2f}")
    for a in axes[r_]:
        a.axis("off")
plt.suptitle("TNUI-2021 val (2nd external cohort): lesion-grounded ROI vs true mask", y=1.0)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "phase3d_qualitative.png", dpi=140, bbox_inches="tight")
plt.show()

## 9. Run summary and hand-off

In [ ]:
run = {"phase": "3d - second external cohort (TNUI-2021 val) + pre-declared pooled sensitivity",
       "seed": SEED, "device": str(DEVICE),
       "protocol": {"threshold_source": "TN5000 val (raw-raw)", "temperature_source": "TN5000 val",
                    "support_episode": "same as Phase 3 (verified vs run_summary.json)",
                    "oracle": "true TNUI masks (not boxes)",
                    "heads_provenance": HEADS_SRC, "test_bank_provenance": TEST_BANK_SRC,
                    "replication_gate": {"tolerance": _hetol, "pooled_ok": POOLED_OK},
                    "declaration": "val cohort + pooled analysis specified before val inspection"},
       "n_val": int(len(y_val)), "val_prevalence_malignant": float(y_val.mean()),
       "val_benign": int((1 - y_val).sum()),
       "seg_qc_val": {"mean_dice": float(np.mean(TNUI_VAL_BANK["dice"])),
                      "empty_mask_rate": float(np.mean(TNUI_VAL_BANK["empty"]))},
       "tables": {"val_main": "phase3d_tnui_val_main_table.csv",
                  "val_paired": "phase3d_paired_tests_val.csv",
                  "shift_3cohort": "phase3d_domain_shift_3cohort.csv",
                  "pooled_main": "phase3d_pooled_main_table.csv",
                  "pooled_paired": "phase3d_pooled_paired_tests.csv"}}
with open(OUTPUT_ROOT / "run_summary.json", "w") as f:
    json.dump(run, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Artifacts:")
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(f"  {p.name:44s} {p.stat().st_size/1024:9.1f} KB")

## 10. What to send back (one by one)
1. §1 manifest block (val n + prevalence + benign count, matched labels, all three audit lines);
2. §4 feature-bank log — TNUI-**val** mean seg Dice vs 0.708 reference + empty-mask rate, and the
   test-bank provenance line;
3. §5 heads provenance line + ERM probs n= lines;
4. §6 replication-gate drift dict + POOLED_OK, then the TNUI-val main table;
5. §7 val paired tests (T1d–T5d) + 3-cohort shift table;
6. §7b pooled main table + pooled paired tests (P1–P5) *if* the gate passed;
7. §8 qualitative description.

**Reading:** with benign n (val) + 5 (test), the pooled family is the first adequately
negative-powered external comparison of the study. Consistent sign/magnitude of
T1c↔T1d, T2c↔T2d, T4c↔T4d across cohorts matters more than any single p-value —
report both cohorts and the pooled sensitivity, each with its own Bonferroni family.